# 📊 Análise de Sentimento - Unidade B

## Objetivo
Analisar a percepção dos pacientes a partir das avaliações públicas.

## Metodologia
- NLP (análise de sentimento)
- análise temporal
- análise de frequência de palavras

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from transformers import pipeline

In [3]:
df_bancarios = pd.read_csv("04_Bancarios_Google_Reviews.csv")

df_bancarios.head()
df_bancarios.info()
df_bancarios.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   unidade     48 non-null     object
 1   autor       48 non-null     object
 2   rating      48 non-null     int64 
 3   data        48 non-null     int64 
 4   comentario  48 non-null     object
dtypes: int64(2), object(3)
memory usage: 2.0+ KB


(48, 5)

In [4]:
df_bancarios["rating"].value_counts().sort_index()

,count
rating,
1,8
3,2
4,3
5,35


In [5]:
df_bancarios["rating"].value_counts(normalize=True).sort_index() * 100

,proportion
rating,
1,16.666667
3,4.166667
4,6.250000
5,72.916667


In [6]:
df_bancarios["data"].value_counts().sort_index()

,count
data,
2021,4
2023,18
2024,9
2025,15
2026,2


In [7]:
media_por_ano = df_bancarios.groupby("data")["rating"].mean().round(2)
media_por_ano

,rating
data,
2021,4.00
2023,4.72
2024,4.33
2025,3.67
2026,3.00


In [8]:
df_bancarios["comentario"] = df_bancarios["comentario"].fillna("Avaliação sem texto")

df_sent = df_bancarios[df_bancarios["comentario"] != "Avaliação sem texto"].copy()

len(df_sent)

22

In [9]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

In [10]:
def analisar_sentimento(texto):
    resultado = sentiment_pipeline(
        str(texto),
        truncation=True,
        max_length=128
    )[0]
    return resultado["label"]

In [11]:
df_critico = df_sent[df_sent["data"].isin([2023, 2024, 2025, 2026])].copy()

len(df_critico)

19

In [12]:
df_critico["sentimento"] = df_critico["comentario"].apply(analisar_sentimento)

df_critico["sentimento"].value_counts(normalize=True) * 100

,proportion
sentimento,
POS,57.894737
NEG,36.842105
NEU,5.263158


In [13]:
stopwords = [
    "de","da","do","das","dos","a","o","as","os","e","é","em","para",
    "com","que","não","na","no","uma","um","mais","foi","por","se",
    "ao","à","às","como","já","eu","ele","ela","me","minha","meu",
    "muito","porque","isso","está","estava","são","ser","tem","pra",
    "pelo","pela","até","fui","era"
]

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    palavras = texto.split()
    palavras = [p for p in palavras if p not in stopwords and len(p) > 2]
    return palavras

In [14]:
df_neg = df_critico[df_critico["sentimento"] == "NEG"].copy()

todas_palavras = []

for comentario in df_neg["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras.extend(palavras)

Counter(todas_palavras).most_common(20)

[('atendimento', 5),
 ('tinha', 4),
 ('ouvido', 2),
 ('apenas', 2),
 ('consulta', 2),
 ('informada', 2),
 ('horário', 2),
 ('telefone', 2),
 ('novamente', 2),
 ('essa', 2),
 ('médico', 2),
 ('nao', 2),
 ('iria', 2),
 ('atender', 2),
 ('atendentes', 1),
 ('atendeu', 1),
 ('devido', 1),
 ('embora', 1),
 ('outra', 1),
 ('tenha', 1)]

In [15]:
df_pos = df_critico[df_critico["sentimento"] == "POS"].copy()

todas_palavras_pos = []

for comentario in df_pos["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras_pos.extend(palavras)

Counter(todas_palavras_pos).most_common(20)

[('atendimento', 8),
 ('ótimo', 3),
 ('excelente', 3),
 ('dra', 3),
 ('clínica', 2),
 ('atenciosa', 2),
 ('ainda', 2),
 ('educada', 2),
 ('fiquei', 1),
 ('extremamente', 1),
 ('satisfeito', 1),
 ('otorrino', 1),
 ('jamille', 1),
 ('profissional', 1),
 ('incrível', 1),
 ('prestativa', 1),
 ('demonstra', 1),
 ('profundo', 1),
 ('conhecimento', 1),
 ('sua', 1)]

In [16]:
anos = media_por_ano.index.values
medias = media_por_ano.values

coef = np.polyfit(anos, medias, 1)

pred_2027 = np.polyval(coef, 2027)
pred_2028 = np.polyval(coef, 2028)

pred_2027, pred_2028

(np.float64(3.2724324324323675), np.float64(3.0625675675674984))